In [2]:
%load_ext autoreload
%autoreload 2

In [4]:
from pathlib import Path
import importlib
import logging
import sys

PROJECT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

import src.pipeline_rf_Copy1_med180 as rf_med180
rf_med180 = importlib.reload(rf_med180)

from src.config_rf_pipeline_Copy1_med180 import load_config

cfg = load_config(PROJECT / "configs" / "RandomForest_260507_Copy1.json")

log_path = PROJECT / "logs" / "rf_pipeline_260507_Copy1_med180.log"
log_path.parent.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    filename=str(log_path),
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)

logger = logging.getLogger("RandomForest_260507_Copy1_med180")
logger.info("Inicio pipeline RF con variables med180 y normalización Min-Max por fold")

print("Módulo cargado desde:")
print(rf_med180.__file__)

Módulo cargado desde:
C:\Users\Equipo\Tesis\SemestreIV\Objetivo1\Objetivo1_26_02_20\src\pipeline_rf_Copy1_med180.py


In [6]:
print("enable_buffer_features:", cfg.enable_buffer_features)
print("buffer_radius_m:", cfg.buffer_radius_m)
print("buffer_statistic:", cfg.buffer_statistic)
print("epsg_work:", cfg.epsg_work)

print("\nBandas Landsat 8 puntuales:")
print(rf_med180.L8_RAW_FEATURES)

print("\nBandas Sentinel-1 puntuales:")
print(rf_med180.S1_RAW_FEATURES)

print("\nVariables Landsat 8 med180:")
print(rf_med180.L8_BUFFER_FEATURES)

print("\nVariables Sentinel-1 med180:")
print(rf_med180.S1_BUFFER_FEATURES)

print("\nVariables crudas activas:")
print(rf_med180.get_raw_features(cfg.enable_buffer_features))

print("\nVariables normalizadas activas:")
print(rf_med180.get_norm_features(cfg.enable_buffer_features))

enable_buffer_features: True
buffer_radius_m: 180.0
buffer_statistic: median
epsg_work: 9377

Bandas Landsat 8 puntuales:
['SR_B5', 'SR_B6', 'SR_B7', 'NDVI', 'EVI', 'NBR', 'NDWI']

Bandas Sentinel-1 puntuales:
['VV', 'VH', 'angle', 'VVVH_ratio', 'VV_Difference', 'VH_Difference', 'VHVV_Difference']

Variables Landsat 8 med180:
['SR_B5_med180', 'SR_B6_med180', 'SR_B7_med180', 'NDVI_med180', 'EVI_med180', 'NBR_med180', 'NDWI_med180']

Variables Sentinel-1 med180:
['VV_med180', 'VH_med180', 'angle_med180', 'VVVH_ratio_med180', 'VV_Difference_med180', 'VH_Difference_med180', 'VHVV_Difference_med180']

Variables crudas activas:
['SR_B5', 'SR_B6', 'SR_B7', 'NDVI', 'EVI', 'NBR', 'NDWI', 'VV', 'VH', 'angle', 'VVVH_ratio', 'VV_Difference', 'VH_Difference', 'VHVV_Difference', 'SR_B5_med180', 'SR_B6_med180', 'SR_B7_med180', 'NDVI_med180', 'EVI_med180', 'NBR_med180', 'NDWI_med180', 'VV_med180', 'VH_med180', 'angle_med180', 'VVVH_ratio_med180', 'VV_Difference_med180', 'VH_Difference_med180', 'VHVV_D

# PRUEBA #

In [8]:
preflight = rf_med180.run_preflight_med180_ab(
    cfg=cfg,
    logger=logger,
    max_points=5,
)

print("Forma de g_master de prueba:", preflight["g_master"].shape)

print("\nVariables Sentinel-1 med180:")
display(preflight["g_master"][rf_med180.S1_BUFFER_FEATURES])

print("\nVariables Landsat 8 med180:")
display(preflight["g_master"][rf_med180.L8_BUFFER_FEATURES])

print("\nNaN por variable en la prueba:")
display(preflight["nan_counts"].to_frame("missing_n"))

C:\Users\Equipo\Tesis\SemestreIV\Objetivo1\Objetivo1_26_02_20\src\points_loader.py:76: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  g = pd.concat([g_fire, g_abs], ignore_index=True)


Forma de g_master de prueba: (5, 89)

Variables Sentinel-1 med180:


,VV_med180,VH_med180,angle_med180,VVVH_ratio_med180,VV_Difference_med180,VH_Difference_med180,VHVV_Difference_med180
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Variables Landsat 8 med180:


,SR_B5_med180,SR_B6_med180,SR_B7_med180,NDVI_med180,EVI_med180,NBR_med180,NDWI_med180
0,0.264860,0.133602,0.058088,0.805905,0.444618,0.364069,-0.748026
1,0.300720,0.153155,0.072498,0.797163,0.473642,0.337744,-0.742171
2,0.211689,0.129684,0.091335,0.565715,0.256202,0.234024,-0.551160
3,0.135129,0.121221,0.086275,0.475563,0.177521,0.071350,-0.502786
4,0.312119,0.237594,0.139652,0.609595,0.397314,0.131759,-0.591337



NaN por variable en la prueba:


,missing_n
VHVV_Difference_med180,5
angle,5
VV_med180,5
VHVV_Difference,5
VH_Difference,5
VV_Difference,5
VVVH_ratio,5
VH,5
VV,5
VH_med180,5


In [12]:
import importlib
import src.pipeline_rf_Copy1_med180 as rf_med180

rf_med180 = importlib.reload(rf_med180)

# =========================================================
# CARGAR PUNTOS Y ASIGNAR FECHAS A AUSENCIAS
# =========================================================

g_points = rf_med180.load_points(
    path_fire_shp=str(cfg.path_fire_shp),
    path_abs_shp=str(cfg.path_abs_shp),
    date_col=cfg.date_col,
    epsg_work=cfg.epsg_work,
)

g_points = rf_med180.assign_dates_to_absences(
    g_points,
    date_col=cfg.date_col,
    seed=cfg.seed_abs_dates,
)

# =========================================================
# INDEXAR RASTERS
# =========================================================

l8_index = rf_med180.index_l8_rasters(
    str(cfg.l8_dir),
    logger=logger,
)

s1_index = rf_med180.index_s1_rasters_from_filenames(
    str(cfg.s1_dir),
    logger=logger,
)

print("Total puntos cargados:", len(g_points))
print("Escenas Landsat 8 indexadas:", len(l8_index))
print("Escenas Sentinel-1 indexadas:", len(s1_index))

Total puntos cargados: 1922
Escenas Landsat 8 indexadas: 30
Escenas Sentinel-1 indexadas: 113


C:\Users\Equipo\Tesis\SemestreIV\Objetivo1\Objetivo1_26_02_20\src\points_loader.py:76: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  g = pd.concat([g_fire, g_abs], ignore_index=True)


In [14]:
import pandas as pd

diagnostico_s1 = []

for i, row in g_points.head(5).iterrows():
    t0 = row[cfg.date_col]

    sel = rf_med180.select_s1_scene_for_point(
        t0=t0,
        s1_df=s1_index,
        start_offset_days=cfg.s1_start_offset_days,
        window_days=cfg.s1_window_days,
    )

    diagnostico_s1.append({
        "point_id": row["point_id"],
        "fecha_punto": pd.to_datetime(t0),
        "fecha_inicio_busqueda": pd.to_datetime(t0) + pd.Timedelta(
            days=cfg.s1_start_offset_days
        ),
        "fecha_fin_busqueda": pd.to_datetime(t0) + pd.Timedelta(
            days=cfg.s1_window_days
        ),
        "escena_s1_encontrada": sel is not None,
        "fecha_escena_s1": (
            pd.Timestamp(sel["date"]) if sel is not None else pd.NaT
        ),
        "archivo_s1": (
            sel["path"] if sel is not None else None
        ),
    })

diagnostico_s1 = pd.DataFrame(diagnostico_s1)

display(diagnostico_s1)

,point_id,fecha_punto,fecha_inicio_busqueda,fecha_fin_busqueda,escena_s1_encontrada,fecha_escena_s1,archivo_s1
0,1,2014-09-29,2014-09-30,2014-11-28,False,NaT,None
1,2,2014-09-29,2014-09-30,2014-11-28,False,NaT,None
2,3,2014-02-04,2014-02-05,2014-04-05,False,NaT,None
3,4,2014-02-04,2014-02-05,2014-04-05,False,NaT,None
4,5,2014-02-02,2014-02-03,2014-04-03,False,NaT,None


# RESTO DEL CODIGO INICIAL#

In [18]:
res = rf_med180.run_pipeline(cfg, logger)

print("========== MODELOS A Y B ==========")
print("OUT_DIR:", res["out_dir"])
print("Raw features:", res["raw_features"])
print("Features normalizadas:", res["features"])
print("Excel Min-Max:", res["normalization_excel"])

print("\nDataset A normalizado:", res["dfA"].shape)
print("Dataset B normalizado:", res["dfB"].shape)

display(res["coverage"])

print("Parámetros finales Modelo A")
display(res["params_A_final"])

print("Parámetros finales Modelo B")
display(res["params_B_final"])

print("Parámetros por fold Modelo A")
display(res["params_A_folds"].head())

print("Parámetros por fold Modelo B")
display(res["params_B_folds"].head())

display(res["foldsA"].head())
display(res["foldsB"].head())
display(res["dfA"].head())
display(res["dfB"].head())

C:\Users\Equipo\Tesis\SemestreIV\Objetivo1\Objetivo1_26_02_20\src\points_loader.py:76: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  g = pd.concat([g_fire, g_abs], ignore_index=True)


=== DEBUG PRE-CV ===
g_points: (1922, 51)
g_master: (1922, 89)
FEATURES: ['SR_B5_norm', 'SR_B6_norm', 'SR_B7_norm', 'NDVI_norm', 'EVI_norm', 'NBR_norm', 'NDWI_norm', 'VV_norm', 'VH_norm', 'angle_norm', 'VVVH_ratio_norm', 'VV_Difference_norm', 'VH_Difference_norm', 'VHVV_Difference_norm', 'SR_B5_med180_norm', 'SR_B6_med180_norm', 'SR_B7_med180_norm', 'NDVI_med180_norm', 'EVI_med180_norm', 'NBR_med180_norm', 'NDWI_med180_norm', 'VV_med180_norm', 'VH_med180_norm', 'angle_med180_norm', 'VVVH_ratio_med180_norm', 'VV_Difference_med180_norm', 'VH_Difference_med180_norm', 'VHVV_Difference_med180_norm']

Conteo label en g_master:
label
1    961
0    961
Name: count, dtype: int64

L8 found:
l8_found
True     1055
False     867
Name: count, dtype: int64

S1 found:
s1_found
False    1292
True      630
Name: count, dtype: int64

NaN por feature cruda en g_master:
SR_B5                     1222
SR_B6                     1222
SR_B7                     1222
NDVI                      1222
EVI          

ValueError: dfA quedó vacío antes de CV: ningún punto cumple caso completo en todas las variables puntuales y med180. Revisa la cobertura L8/S1.

Variables con mayor número de NaN:
VHVV_Difference_med180    1922
VH_Difference_med180      1922
VV_Difference_med180      1922
VVVH_ratio_med180         1922
angle_med180              1922
VH_med180                 1922
VV_med180                 1922
SR_B6_med180              1809
NDWI_med180               1809
NBR_med180                1809
EVI_med180                1809
NDVI_med180               1809
SR_B7_med180              1809
SR_B5_med180              1809
VVVH_ratio                1370
VH                        1370
VV                        1370
angle                     1366
VHVV_Difference           1321
VH_Difference             1321

Porcentaje de NaN en esas variables:
VHVV_Difference_med180    100.000000
VH_Difference_med180      100.000000
VV_Difference_med180      100.000000
VVVH_ratio_med180         100.000000
angle_med180 